# V10.3: IXI SSL Pretrain (Zero BraTS Overlap) + V7.0 Loss Fusion

**Zero data leakage experiment.** SSL pretrain on IXI (healthy brain MRI), fine-tune on BraTS2020 with text.

| Stage | Task | Data | Labels | Text |
|-------|------|------|--------|------|
| **SSL Pretrain** | Masked reconstruction | IXI (~577 healthy) | **NO** | NO |
| **Fine-tune** | Segmentation | BraTS2020 (295) | YES | **YES** |

Key differences from V10.2:
- **Zero overlap** with BraTS — IXI is a completely independent healthy brain dataset
- Channel duplication `[T1, T1, T2, T2]` for 4-channel encoder compatibility
- V7.0 Boundary + Hierarchy Loss for HD95 improvement
- Tests whether cross-domain SSL (healthy → tumor) can bootstrap text-guided segmentation

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi 2>/dev/null || echo 'No GPU'
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0: break
        if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else: raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS (needed for Stage 2 fine-tune)
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)

def sync_and_tag(tag):
    """Sync checkpoints to Drive with version-specific names."""
    lc = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(lc): return
    for f in glob.glob(os.path.join(lc, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    b = os.path.join(lc, 'best.pth')
    if os.path.exists(b): shutil.copy2(b, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
    l = os.path.join(lc, 'last.pth')
    if os.path.exists(l): shutil.copy2(l, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

def _file_is_stable(path, wait=5):
    """Check if a file hasn't changed size for `wait` seconds."""
    if not os.path.exists(path):
        return False
    s1 = os.path.getsize(path)
    time.sleep(wait)
    if not os.path.exists(path):
        return False
    s2 = os.path.getsize(path)
    return s1 == s2 and s1 > 0

def _auto_sync_loop(stop_event, interval=300):
    """Background thread: sync checkpoints every `interval` seconds."""
    while not stop_event.is_set():
        stop_event.wait(interval)
        if stop_event.is_set():
            break
        lc = os.path.join(REPO_DIR, 'checkpoints')
        if not os.path.exists(lc):
            continue
        for f in glob.glob(os.path.join(lc, '*.pth')):
            if _file_is_stable(f):
                try:
                    shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
                except Exception as e:
                    print(f'[auto-sync] Error copying {f}: {e}')
        print(f'[auto-sync] {time.strftime("%H:%M:%S")} synced')

_stop_sync = threading.Event()
_sync_thread = threading.Thread(target=_auto_sync_loop, args=(_stop_sync, 300), daemon=True)
_sync_thread.start()
print('Background auto-sync started (every 5 min)')

print('Setup complete')

Mounted at /content/drive
Sun Apr 12 08:09:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## Step 1: Download + Preprocess IXI

IXI dataset: ~577 healthy brain MRI (T1 + T2).
- Download T1 and T2 NIfTI tar archives from biomedic.doc.ic.ac.uk
- Match subjects with both T1 and T2
- Resample to 240x240x155 (BraTS standard)
- Normalize per-volume (zero mean, unit std)
- Stack as 4-channel `[T1, T1, T2, T2]` for encoder compatibility
- Split 85/15 train/val, save as `.npz`

In [2]:
import os, sys, tarfile, glob, shutil, time
import numpy as np
os.chdir(REPO_DIR)

IXI_DRIVE_CACHE = os.path.join(DRIVE_BASE, 'IXI_processed')
IXI_LOCAL = '/content/IXI_processed'
TARGET_SHAPE = (240, 240, 155)

# Check if already cached on Drive
if os.path.isdir(IXI_DRIVE_CACHE) and len(glob.glob(os.path.join(IXI_DRIVE_CACHE, 'train', '*', 'data.npz'))) > 400:
    print(f'IXI data cached on Drive ({IXI_DRIVE_CACHE}), copying to local SSD...')
    if os.path.exists(IXI_LOCAL):
        shutil.rmtree(IXI_LOCAL)
    shutil.copytree(IXI_DRIVE_CACHE, IXI_LOCAL)
    n_train = len(glob.glob(os.path.join(IXI_LOCAL, 'train', '*', 'data.npz')))
    n_val = len(glob.glob(os.path.join(IXI_LOCAL, 'val', '*', 'data.npz')))
    print(f'Loaded from cache: train={n_train}, val={n_val}')
else:
    print('IXI data not cached. Downloading and preprocessing...')

    import urllib.request
    import nibabel as nib
    from scipy.ndimage import zoom

    RAW_DIR = '/content/IXI_raw'
    os.makedirs(RAW_DIR, exist_ok=True)

    # Download T1 and T2
    urls = {
        'T1': 'https://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI-T1.tar',
        'T2': 'https://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI-T2.tar',
    }
    for modality, url in urls.items():
        tar_path = os.path.join(RAW_DIR, f'IXI-{modality}.tar')
        extract_dir = os.path.join(RAW_DIR, modality)
        if os.path.isdir(extract_dir) and len(os.listdir(extract_dir)) > 100:
            print(f'{modality}: already extracted ({len(os.listdir(extract_dir))} files)')
            continue
        if not os.path.exists(tar_path):
            print(f'Downloading {modality} ({url})...')
            t0 = time.time()
            urllib.request.urlretrieve(url, tar_path)
            print(f'  Downloaded in {time.time()-t0:.0f}s ({os.path.getsize(tar_path)/1e9:.1f} GB)')
        print(f'Extracting {modality}...')
        os.makedirs(extract_dir, exist_ok=True)
        with tarfile.open(tar_path, 'r') as tf:
            tf.extractall(extract_dir)
        print(f'  Extracted {len(os.listdir(extract_dir))} files')
        # Remove tar to save space
        os.remove(tar_path)

    # Match subjects with both T1 and T2
    t1_dir = os.path.join(RAW_DIR, 'T1')
    t2_dir = os.path.join(RAW_DIR, 'T2')
    t1_files = {f.split('-')[0]: os.path.join(t1_dir, f) for f in os.listdir(t1_dir) if f.endswith('.nii.gz')}
    t2_files = {f.split('-')[0]: os.path.join(t2_dir, f) for f in os.listdir(t2_dir) if f.endswith('.nii.gz')}
    common_ids = sorted(set(t1_files.keys()) & set(t2_files.keys()))
    print(f'Matched subjects: {len(common_ids)} (T1={len(t1_files)}, T2={len(t2_files)})')

    # Split 85/15
    np.random.seed(42)
    indices = np.random.permutation(len(common_ids))
    n_train = int(len(common_ids) * 0.85)
    train_ids = [common_ids[i] for i in indices[:n_train]]
    val_ids = [common_ids[i] for i in indices[n_train:]]
    print(f'Split: train={len(train_ids)}, val={len(val_ids)}')

    # Process and save
    def process_subject(subj_id, t1_path, t2_path, out_dir):
        """Load T1+T2, resample, normalize, save as 4ch npz."""
        os.makedirs(out_dir, exist_ok=True)
        try:
            t1_img = nib.load(t1_path).get_fdata().astype(np.float32)
            t2_img = nib.load(t2_path).get_fdata().astype(np.float32)
        except Exception as e:
            print(f'  SKIP {subj_id}: {e}')
            return False

        # Resample both to target shape
        for name, img in [('T1', t1_img), ('T2', t2_img)]:
            if img.ndim != 3 or min(img.shape) < 10:
                print(f'  SKIP {subj_id}: bad shape {img.shape}')
                return False

        zoom_t1 = [t / s for t, s in zip(TARGET_SHAPE, t1_img.shape)]
        zoom_t2 = [t / s for t, s in zip(TARGET_SHAPE, t2_img.shape)]
        t1_resampled = zoom(t1_img, zoom_t1, order=1)
        t2_resampled = zoom(t2_img, zoom_t2, order=1)

        # Normalize per-volume (zero mean, unit std in foreground)
        for vol in [t1_resampled, t2_resampled]:
            fg = vol > vol.mean() * 0.1
            if fg.sum() > 100:
                mu = vol[fg].mean()
                sigma = vol[fg].std() + 1e-8
                vol[:] = (vol - mu) / sigma

        # Stack as [T1, T1, T2, T2] — 4 channels for ModalityGroupPatchEmbed3D
        image = np.stack([t1_resampled, t1_resampled, t2_resampled, t2_resampled], axis=0)
        np.savez_compressed(os.path.join(out_dir, 'data.npz'), image=image)
        return True

    os.makedirs(IXI_LOCAL, exist_ok=True)
    for split_name, id_list in [('train', train_ids), ('val', val_ids)]:
        split_dir = os.path.join(IXI_LOCAL, split_name)
        os.makedirs(split_dir, exist_ok=True)
        success = 0
        for i, sid in enumerate(id_list):
            out = os.path.join(split_dir, sid)
            if os.path.exists(os.path.join(out, 'data.npz')):
                success += 1
                continue
            ok = process_subject(sid, t1_files[sid], t2_files[sid], out)
            if ok:
                success += 1
            if (i + 1) % 50 == 0:
                print(f'  {split_name}: {i+1}/{len(id_list)} processed ({success} ok)')
        print(f'{split_name}: {success}/{len(id_list)} subjects saved')

    # Cache to Drive
    print('Caching processed IXI to Drive...')
    if os.path.exists(IXI_DRIVE_CACHE):
        shutil.rmtree(IXI_DRIVE_CACHE)
    shutil.copytree(IXI_LOCAL, IXI_DRIVE_CACHE)
    print('Cached to Drive.')

    # Cleanup raw
    shutil.rmtree(RAW_DIR, ignore_errors=True)

# Final counts
n_train = len(glob.glob(os.path.join(IXI_LOCAL, 'train', '*', 'data.npz')))
n_val = len(glob.glob(os.path.join(IXI_LOCAL, 'val', '*', 'data.npz')))
print(f'IXI ready: train={n_train}, val={n_val}, total={n_train+n_val}')
print(f'Local path: {IXI_LOCAL}')

IXI data not cached. Downloading and preprocessing...
  Downloaded in 150s (4.8 GB)
Extracting T1...


/tmp/ipykernel_4039/1088280463.py:47: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(extract_dir)


  Extracted 581 files
  Downloaded in 200s (3.9 GB)
Extracting T2...
  Extracted 578 files
Matched subjects: 577 (T1=581, T2=578)
Split: train=490, val=87
  train: 50/490 processed (50 ok)
  train: 100/490 processed (100 ok)
  train: 150/490 processed (150 ok)
  train: 200/490 processed (200 ok)
  train: 250/490 processed (250 ok)
  train: 300/490 processed (300 ok)
  train: 350/490 processed (350 ok)
  train: 400/490 processed (400 ok)
  train: 450/490 processed (450 ok)
train: 490/490 subjects saved
  val: 50/87 processed (50 ok)
val: 87/87 subjects saved
Caching processed IXI to Drive...
Cached to Drive.
IXI ready: train=490, val=87, total=577
Local path: /content/IXI_processed


## Step 2: SSL Pretrain on IXI (200 epochs)

Masked Image Modeling on IXI healthy brains. 50% patch masking, L1 reconstruction.
Backbone learns normal brain anatomy from a completely independent dataset.

In [3]:
import os, shutil, torch
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

def get_ssl_epoch(path):
    if not os.path.exists(path):
        return -1
    try:
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        return ckpt.get('epoch', -1)
    except Exception as e:
        print(f'Could not read {path}: {e}')
        return -1

drive_ssl_last = os.path.join(DRIVE_CKPT, 'last_ssl_ixi.pth')
drive_ssl_best = os.path.join(DRIVE_CKPT, 'best_ssl_ixi.pth')

last_epoch = get_ssl_epoch(drive_ssl_last)
print(f'SSL IXI last checkpoint on Drive: epoch={last_epoch}')

if last_epoch >= 199:
    print(f'SSL pretrain already complete (epoch {last_epoch}). Skip to Stage 2.')
else:
    os.makedirs('checkpoints', exist_ok=True)
    resume_arg = ''
    if last_epoch >= 0:
        shutil.copy2(drive_ssl_last, 'checkpoints/last_ssl.pth')
        if os.path.exists(drive_ssl_best):
            shutil.copy2(drive_ssl_best, 'checkpoints/best_ssl.pth')
        resume_arg = '--resume checkpoints/last_ssl.pth'
        print(f'Resuming SSL pretrain from epoch {last_epoch}')
    else:
        print('Starting SSL pretrain fresh on IXI')

    !python -u scripts/pretrain_ssl.py \
        --data-dir {IXI_LOCAL} \
        --dataset ixi \
        --epochs 200 \
        --batch-size 4 \
        --lr 1e-4 \
        --mask-ratio 0.5 \
        --embed-dim 48 \
        {resume_arg}

    # Sync with version-specific names
    for src_name, dst_name in [('best_ssl.pth', 'best_ssl_ixi.pth'),
                                ('last_ssl.pth', 'last_ssl_ixi.pth')]:
        src = os.path.join('checkpoints', src_name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_CKPT, dst_name))
    print('SSL IXI pretrain complete!')

SSL IXI last checkpoint on Drive: epoch=-1
Starting SSL pretrain fresh on IXI
/content/TextMamba3D/models/__init__.py:1: UserWarning: Real Mamba3 not available, falling back to Mamba2.
  from .mamba_block import (
Device: cuda
Training samples: 490
/content/TextMamba3D/models/mamba_block.py:357: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.dhw_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:358: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.hwd_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:359: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.wdh_fwd = _create_ssm(**ssm_kw)
SSL Pretraining: 200 epochs, mask_ratio=0.5
Encoder params: 9,394,982
Epoch 0:  44% 54/122 [03:36<01:13,  1.08s/it, loss=0.3792, lr=1.00e-05]  [auto-sync] 10:02:41 synced
Epoch 0: 100% 122/122 [05

In [4]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
lc = os.path.join(REPO_DIR, 'checkpoints')
fs = glob.glob(os.path.join(lc, '*.pth'))
if fs:
    for f in sorted(fs):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    subprocess.run(['sync'], check=True)
    print(f'{len(fs)} files synced')
else:
    print('No checkpoints')

2 files synced


## Step 3: Fine-tune with Text on BraTS2020

Load IXI SSL-pretrained encoder into TextMamba3D, then fine-tune with:
- SeqCA text fusion (text guides segmentation)
- V7.0 Boundary Loss (improve HD95)
- V7.0 Hierarchy Loss (WT > TC > ET structure)

In [ ]:
import os, glob, torch, shutil
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)

V103_BEST = os.path.join(DRIVE_CKPT, 'best_V10.3.pth')
V103_LAST = os.path.join(DRIVE_CKPT, 'last_V10.3.pth')
SSL_CKPT = os.path.join(DRIVE_CKPT, 'best_ssl_ixi.pth')

# Check if SSL is newer than existing V10.3 (means SSL was re-trained)
need_retrain = False
if os.path.exists(V103_BEST) and os.path.exists(SSL_CKPT):
    ssl_time = os.path.getmtime(SSL_CKPT)
    v103_time = os.path.getmtime(V103_BEST)
    if ssl_time > v103_time:
        print('SSL IXI checkpoint is NEWER than V10.3 -- need to re-run Stage 2')
        os.remove(V103_BEST)
        if os.path.exists(V103_LAST):
            os.remove(V103_LAST)
        need_retrain = True
    else:
        print(f'V10.3 already complete: {V103_BEST}')
        print('Skip to evaluation')
elif os.path.exists(V103_BEST):
    print(f'V10.3 already complete: {V103_BEST}')
    print('Skip to evaluation')
else:
    need_retrain = True

if need_retrain:
    if os.path.exists(V103_LAST):
        # Resume interrupted Stage 2
        shutil.copy2(V103_LAST, os.path.join(ckpt_dir, 'last.pth'))
        print(f'Resuming V10.3 Stage 2 from: {V103_LAST}')
        !python -u train.py \
            --config configs/autoresearch/V10.3_ixi_ssl.yaml \
            --resume checkpoints/last.pth \
            --no-text-ratio 0.15 \
            --grad-accum 2
        sync_and_tag('V10.3')
        print('V10.3 Stage 2 complete!')
    else:
        # First run: create SSL-initialized checkpoint from IXI pretrain
        assert os.path.exists(SSL_CKPT), f'SSL IXI checkpoint not found: {SSL_CKPT}'
        ssl_state = torch.load(SSL_CKPT, map_location='cpu', weights_only=False)
        encoder_state = ssl_state['encoder']

        from models.textmamba3d import TextMamba3D
        model = TextMamba3D(
            img_size=(128,128,128), embed_dim=48, depths=[2,2,2,2],
            text_embed_dim=256, use_mamba3=True, headdim=48,
            fusion_type='seqca',
        )
        full_state = model.state_dict()
        loaded = 0
        for k, v in encoder_state.items():
            full_key = f'img_encoder.{k}'
            if full_key in full_state and full_state[full_key].shape == v.shape:
                full_state[full_key] = v
                loaded += 1
        print(f'Loaded {loaded} encoder params from IXI SSL checkpoint')

        for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
            os.remove(f)
        ssl_resume = os.path.join(ckpt_dir, 'ssl_init.pth')
        torch.save({'model': full_state, 'epoch': -1, 'best_dice': 0, 'best_dice_no_text': 0}, ssl_resume)

        print('Stage 2: Fine-tune with text (from IXI 200-epoch SSL + V7.0 loss)')
        !python -u train.py \
            --config configs/autoresearch/V10.3_ixi_ssl.yaml \
            --resume "{ssl_resume}" \
            --reset-optimizer \
            --no-text-ratio 0.15 \
            --grad-accum 2
        sync_and_tag('V10.3')
        print('V10.3 Stage 2 complete!')

/content/TextMamba3D/models/__init__.py:1: UserWarning: Real Mamba3 not available, falling back to Mamba2.
  from .mamba_block import (
/content/TextMamba3D/models/mamba_block.py:357: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.dhw_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:358: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.hwd_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:359: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.wdh_fwd = _create_ssm(**ssm_kw)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in t

[auto-sync] 19:32:46 synced
[auto-sync] 19:37:56 synced
[auto-sync] 19:43:07 synced
[auto-sync] 19:48:18 synced
[auto-sync] 19:53:28 synced
[auto-sync] 19:58:39 synced
[auto-sync] 20:03:50 synced
[auto-sync] 20:09:01 synced
[auto-sync] 20:14:11 synced
[auto-sync] 20:19:22 synced
[auto-sync] 20:24:33 synced
[auto-sync] 20:29:43 synced
[auto-sync] 20:34:54 synced
[auto-sync] 20:40:05 synced
[auto-sync] 20:45:15 synced
[auto-sync] 20:50:26 synced
[auto-sync] 20:55:37 synced
[auto-sync] 21:00:47 synced
[auto-sync] 21:05:58 synced
[auto-sync] 21:11:09 synced
[auto-sync] 21:16:20 synced
[auto-sync] 21:21:30 synced
[auto-sync] 21:26:41 synced
[auto-sync] 21:31:52 synced
[auto-sync] 21:37:02 synced
[auto-sync] 21:42:13 synced
[auto-sync] 21:47:24 synced
[auto-sync] 21:52:35 synced
[auto-sync] 21:57:45 synced
[auto-sync] 22:02:56 synced
[auto-sync] 22:08:07 synced
[auto-sync] 22:13:18 synced
[auto-sync] 22:18:28 synced
[auto-sync] 22:23:39 synced
[auto-sync] 22:28:50 synced
[auto-sync] 22:34:00

## Evaluation

In [ ]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V10.3.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V10.3_ixi_ssl.yaml'
results = {}
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG, '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

# Stop auto-sync thread
_stop_sync.set()
print()
print('Comparison table:')
print('  V5.0  (scratch, SeqCA):       Mean=0.8479, delta=+0.55%')
print('  V8.0  (sup pretrain):         Mean=0.8753, delta=0.00%')
print('  V10.2 (BraTS2021 SSL+text):   Mean=?, delta=?')
print('  V10.3 (IXI SSL+text+V7.0):    Mean=?, delta=?')
print('  TextBraTS (SwinUNETR+IN):     Mean=0.853, delta=+1.5%')
print()
print('V10.3 hypothesis: cross-domain SSL (healthy brain) + V7.0 loss')
print('should match or beat V10.2 while proving zero-leakage viability.')